<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 4 &nbsp;&middot;&nbsp; Feedforward Neural Networks</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>Building, training, and evaluating feedforward neural networks in PyTorch &mdash; from digit recognition to predicting customer churn.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| 4.1 FFN Architecture | Layers and neurons · `nn.Module` · Counting parameters · Design decisions |
| 4.2 Warm-Up: MNIST Digit Recognition | Loading data · Defining and training an FFN · Evaluation · Confusion matrix |
| 4.3 Evaluation Metrics for Classification | Accuracy · Precision · Recall · F1 · When accuracy is not enough |
| 4.4 Business Application: Customer Churn | EDA · Class imbalance · FFN classifier · Full evaluation suite |
| 4.5 Pipeline: Saving and Deploying the Churn Model | Applying `ModelPipeline` from Chapter 3 · Save · Load · Predict · Retrain |

> **Datasets used in this chapter:**
> - **MNIST** — loaded directly via `torchvision.datasets` (no download required)
> - **Telco Customer Churn** — downloaded from Kaggle in Section 4.4

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 4 — Setup
# Run this cell first. It installs all packages needed for this chapter
# and sets consistent plot styling.
# Expected time: under 2 minutes on Colab.
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet torch torchvision numpy pandas matplotlib seaborn scikit-learn kaggle tqdm dill

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import torchvision
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_fscore_support
)

from tqdm import tqdm

# Detect the best available compute device
device = (
    torch.device('cuda')  if torch.cuda.is_available()                              else
    torch.device('mps')   if hasattr(torch.backends, 'mps') and
                             torch.backends.mps.is_available()                       else
    torch.device('cpu')
)

# Consistent plot style
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f9fa'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.alpha']       = 0.4
plt.rcParams['font.size']        = 11

print(f"Setup complete. Running on: {device}")
print("Ready to begin Chapter 4.")

---
# 4.1 FFN Architecture

## What a feedforward network actually does

A **feedforward neural network (FFN)** — sometimes called a multi-layer perceptron (MLP) — is the foundational architecture in deep learning. Every more specialised architecture you will encounter in later chapters (CNNs, LSTMs, Transformers) contains FFN components at its core.

The term *feedforward* describes the direction data moves: from input, through one or more hidden layers, to output — always forward, never looping back. This distinguishes FFNs from the recurrent networks covered in Chapter 6, where information can cycle.

Each layer in an FFN is a **fully connected** (or *dense*) layer: every neuron in that layer receives input from every neuron in the previous layer. In PyTorch, this is `nn.Linear`.

## Designing an FFN: the key decisions

Before writing any code, three architectural choices must be made. These choices are not derived mathematically — they are informed by experience, the nature of the data, and experimentation.

| Decision | Guidance |
|----------|----------|
| **How many layers?** | Start with 2–3 hidden layers. More layers capture more complex patterns but are harder to train and slower to converge. |
| **How many neurons per layer?** | A common starting point is 64–512 neurons per layer. Wider layers memorise more; narrower layers generalise better. Start wider, add regularisation if it overfits. |
| **What activation function?** | ReLU for hidden layers — it trains fast and avoids the vanishing gradient problem. For the output layer: Sigmoid for binary classification, Softmax for multi-class, no activation for regression. |

## Defining a network in PyTorch

PyTorch uses a class-based approach. Every network is a Python class that inherits from `nn.Module`. Two methods are required:
- `__init__`: define the layers
- `forward`: describe how data flows through them

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Defining an FFN in PyTorch
#
# This generic FFN accepts:
#   input_size  — number of input features
#   hidden_sizes — list of neuron counts, one per hidden layer
#   output_size — number of output neurons
#   dropout_p   — dropout probability (0 = no dropout)
#
# We will reuse this same class for both MNIST and the churn problem.
# ─────────────────────────────────────────────────────────────────────────────

class FFN(nn.Module):
    """
    A flexible feedforward neural network.
    Stacks fully-connected layers with ReLU activation and optional dropout.
    """

    def __init__(self, input_size, hidden_sizes, output_size, dropout_p=0.3):
        super().__init__()

        # Build the hidden layers dynamically from the hidden_sizes list
        layers = []
        prev_size = input_size

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))  # fully connected layer
            layers.append(nn.ReLU())                           # non-linearity
            layers.append(nn.Dropout(p=dropout_p))             # regularisation
            prev_size = hidden_size

        # Output layer — no activation here; loss function handles it
        layers.append(nn.Linear(prev_size, output_size))

        # nn.Sequential chains all layers so we can call them in order
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # Data flows straight through — this is the 'feedforward' part
        return self.network(x)


# ── Inspect the architecture before training anything ──────────────────────

def count_parameters(model):
    """Count trainable parameters — a quick proxy for model complexity."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Three architectures for comparison
configs = [
    {'hidden_sizes': [64],          'label': 'Shallow  (1 hidden layer)'},
    {'hidden_sizes': [128, 64],     'label': 'Medium   (2 hidden layers)'},
    {'hidden_sizes': [256, 128, 64],'label': 'Deep     (3 hidden layers)'},
]

print(f"{'Architecture':<32} {'Layers':<10} {'Parameters':>12}")
print("-" * 56)

for cfg in configs:
    model = FFN(input_size=784, hidden_sizes=cfg['hidden_sizes'], output_size=10)
    n_params = count_parameters(model)
    n_layers  = len(cfg['hidden_sizes'])
    print(f"  {cfg['label']:<30} {n_layers:<10} {n_params:>12,}")

print()
print("Note: input_size=784 (28×28 MNIST pixels), output_size=10 (digits 0–9).")
print("More parameters = more capacity to learn, but also more risk of overfitting.")

### 📝 Exercise 4.1 — Design Your Own Architecture

A retail bank wants to classify loan applications into three risk tiers: Low, Medium, High. The application form captures 12 numeric features (income, age, loan amount, existing debts, etc.).

Fill in the blanks in the code cell below. Think through each choice before writing the number.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 4.1 — Design a network for the loan risk problem
#
# Fill in the values marked with ?.
# ─────────────────────────────────────────────────────────────────────────────

loan_model = FFN(
    input_size   = ?,    # How many input features does the form have?
    hidden_sizes = [?, ?],  # Choose neuron counts for 2 hidden layers
    output_size  = ?,    # How many risk tiers are there?
    dropout_p    = 0.3
)

print(f"Loan risk model — parameters: {count_parameters(loan_model):,}")
print(loan_model)

# Suggested answer (uncomment to reveal):
#
# loan_model = FFN(
#     input_size   = 12,
#     hidden_sizes = [64, 32],
#     output_size  = 3,
#     dropout_p    = 0.3
# )

print("\nExercise 4.1 complete.")

---
# 4.2 Warm-Up: MNIST Digit Recognition

## Why MNIST?

MNIST is a dataset of 70,000 handwritten digits (0–9), each a 28×28 grayscale image. It has been a standard benchmark in deep learning since the 1990s. For our purposes it serves as a clean, well-understood problem to verify that the FFN class we just defined works correctly — before applying it to messier business data.

MNIST is a **multi-class classification** problem: the network receives 784 pixel values (28×28 flattened) and must predict which of 10 digit classes the image belongs to.

## Loading MNIST

`torchvision.datasets` handles MNIST automatically — downloading, caching, and wrapping it in a format PyTorch's `DataLoader` can use. No Kaggle account needed.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load MNIST
#
# transforms.ToTensor()     — converts PIL image to a float tensor in [0, 1]
# transforms.Normalize()    — standardises pixel values to mean=0.5, std=0.5
#                             this helps training converge faster
# ─────────────────────────────────────────────────────────────────────────────

mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # mean, std — one channel (grayscale)
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True,  download=True, transform=mnist_transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=mnist_transform
)

# DataLoaders batch and shuffle the data during training
# shuffle=True on training data prevents the model from learning the order of examples
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

print(f"Training samples:   {len(train_dataset):,}")
print(f"Test samples:       {len(test_dataset):,}")
print(f"Image shape:        {train_dataset[0][0].shape}   (channels, height, width)")
print(f"Classes:            {train_dataset.classes}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualise a sample of training images
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
fig.patch.set_facecolor('white')

for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    # img is (1, 28, 28) — squeeze to (28, 28) for display
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label), fontsize=10)
    ax.axis('off')

fig.suptitle('Figure 4.1 — Sample MNIST digits (label shown above each image)',
             fontsize=11, fontweight='bold', color='#1F3864', y=1.04)
plt.tight_layout()
plt.show()

## Training the MNIST classifier

The training loop follows the same structure introduced conceptually in Chapter 1 and practised in Chapter 3. The only new element here is that we are now classifying across 10 classes rather than predicting a single number.

**Loss function for multi-class classification: `CrossEntropyLoss`**

`nn.CrossEntropyLoss` combines a Softmax activation and the cross-entropy loss into one operation. This is why the output layer of our `FFN` has no activation — PyTorch handles it internally in the loss function. This is a deliberate design choice in PyTorch for numerical stability.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Instantiate the MNIST model
#
# Input:  784 pixels (28×28 flattened)
# Hidden: two layers of 256 and 128 neurons
# Output: 10 neurons, one per digit class
# ─────────────────────────────────────────────────────────────────────────────

mnist_model = FFN(
    input_size   = 784,
    hidden_sizes = [256, 128],
    output_size  = 10,
    dropout_p    = 0.3
).to(device)

# Loss function and optimiser
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mnist_model.parameters(), lr=1e-3)

print(mnist_model)
print(f"\nTotal trainable parameters: {count_parameters(mnist_model):,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Training loop
#
# Each epoch:
#   1. Set model to training mode (enables dropout)
#   2. Loop over batches from the DataLoader
#   3. Flatten images from (B, 1, 28, 28) to (B, 784)
#   4. Forward pass → compute loss → backpropagate → update weights
#
# After each epoch, evaluate on the test set (no gradient computation needed).
# ─────────────────────────────────────────────────────────────────────────────

NUM_EPOCHS = 10

history = {'train_loss': [], 'test_loss': [], 'test_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Training phase ──────────────────────────────────────────────────────
    mnist_model.train()          # activates dropout
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.view(images.size(0), -1).to(device)  # flatten
        labels = labels.to(device)

        optimizer.zero_grad()        # clear gradients from previous step
        outputs = mnist_model(images)  # forward pass
        loss = criterion(outputs, labels)  # compute loss
        loss.backward()              # backpropagate
        optimizer.step()             # update weights

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    # ── Evaluation phase ────────────────────────────────────────────────────
    mnist_model.eval()           # deactivates dropout
    test_loss = 0.0
    correct   = 0
    total     = 0

    with torch.no_grad():        # no gradient computation during evaluation
        for images, labels in test_loader:
            images  = images.view(images.size(0), -1).to(device)
            labels  = labels.to(device)
            outputs = mnist_model(images)
            loss    = criterion(outputs, labels)
            test_loss += loss.item()

            # Predicted class = index of the highest output score
            predicted = outputs.argmax(dim=1)
            correct  += (predicted == labels).sum().item()
            total    += labels.size(0)

    test_loss /= len(test_loader)
    test_acc   = 100.0 * correct / total

    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(f"Epoch {epoch:>2}/{NUM_EPOCHS}  "
          f"Train Loss: {train_loss:.4f}  "
          f"Test Loss: {test_loss:.4f}  "
          f"Test Accuracy: {test_acc:.2f}%")

print("\nTraining complete.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.2 — Training and test loss curves + accuracy
# ─────────────────────────────────────────────────────────────────────────────

epochs_range = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('white')

# Loss curves
ax1.plot(epochs_range, history['train_loss'], color='#2980b9', lw=2.5, label='Train loss')
ax1.plot(epochs_range, history['test_loss'],  color='#e74c3c', lw=2.5, linestyle='--', label='Test loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Figure 4.2a — Loss curves', fontweight='bold', color='#1F3864')
ax1.legend()

# Accuracy
ax2.plot(epochs_range, history['test_acc'], color='#27ae60', lw=2.5, marker='o', markersize=5)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Figure 4.2b — Test accuracy', fontweight='bold', color='#1F3864')
ax2.set_ylim(90, 100)
ax2.axhline(y=history['test_acc'][-1], color='#27ae60', lw=1, linestyle=':', alpha=0.6)
ax2.text(NUM_EPOCHS * 0.6, history['test_acc'][-1] + 0.3,
         f"Final: {history['test_acc'][-1]:.2f}%", color='#27ae60', fontsize=10)

plt.tight_layout()
plt.show()

## The Confusion Matrix

Accuracy tells you how often the model is right overall, but it hides *where* the model struggles. A **confusion matrix** shows the full picture: for every actual class, how many examples were predicted as each possible class?

On the diagonal: correct predictions. Off the diagonal: mistakes. The colour intensity reveals which digits the model most commonly confuses — typically digit pairs that look similar (4 and 9, 3 and 8, 7 and 1).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Confusion matrix on the full test set
# ─────────────────────────────────────────────────────────────────────────────

all_preds  = []
all_labels = []

mnist_model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.size(0), -1).to(device)
        preds  = mnist_model(images).argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=range(10), yticklabels=range(10),
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Count'}
)
ax.set_xlabel('Predicted digit', fontsize=12)
ax.set_ylabel('Actual digit', fontsize=12)
ax.set_title('Figure 4.3 — Confusion Matrix: MNIST Test Set\n'
             'Diagonal = correct · Off-diagonal = errors · Colour = frequency',
             fontsize=12, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()

# Identify the most common confusion pairs
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
top_errors = []
for _ in range(5):
    idx = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
    top_errors.append((idx[0], idx[1], cm_no_diag[idx]))
    cm_no_diag[idx] = 0

print("Top-5 most common errors (actual → predicted):")
for actual, pred, count in top_errors:
    print(f"  Digit {actual} predicted as {pred}: {count} times")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualise misclassified examples
# Shows actual images where the model was wrong — useful for understanding
# what the model finds difficult.
# ─────────────────────────────────────────────────────────────────────────────

misclassified = [
    (img, true, pred)
    for img, true, pred in zip(test_dataset.data, all_labels, all_preds)
    if true != pred
]

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
fig.patch.set_facecolor('white')

for i, ax in enumerate(axes.flat):
    if i < len(misclassified):
        img, true_label, pred_label = misclassified[i]
        ax.imshow(img, cmap='gray')
        ax.set_title(f"T:{true_label} P:{pred_label}",
                     fontsize=8,
                     color='#c0392b')
    ax.axis('off')

fig.suptitle('Figure 4.4 — Misclassified examples  (T = true label, P = predicted)',
             fontsize=11, fontweight='bold', color='#1F3864', y=1.04)
plt.tight_layout()
plt.show()

print(f"Total misclassified: {len(misclassified)} out of {len(test_dataset):,} ({100*len(misclassified)/len(test_dataset):.2f}% error rate)")

---
# 4.3 Evaluation Metrics for Classification

## Why accuracy is not enough

MNIST is a well-balanced dataset — each digit appears roughly equally often. In that setting, accuracy is a fair summary of performance.

Most real business problems are not balanced. Consider customer churn: if 85% of customers stay and 15% churn, a model that *always predicts "no churn"* achieves 85% accuracy — without learning anything useful. Accuracy would say this model is excellent. The business would discover the opposite.

Four metrics together give the full picture:

| Metric | Formula | Business meaning |
|--------|---------|------------------|
| **Precision** | TP / (TP + FP) | Of all customers flagged as likely to churn, what fraction actually churned? |
| **Recall** | TP / (TP + FN) | Of all customers who actually churned, what fraction did we catch? |
| **F1 Score** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall — a single number when both matter |
| **ROC-AUC** | Area under the ROC curve | Overall ability to separate churners from non-churners, regardless of threshold |

## The precision–recall trade-off

Precision and recall pull against each other. To catch every churner (high recall), you must flag more customers — including some who would not have churned (lower precision). To be confident about every flag (high precision), you must flag fewer — missing some actual churners (lower recall).

The right balance depends on the business cost of each error:
- **False negative** (missing a churner): the company loses a customer
- **False positive** (flagging a loyal customer): the company spends on unnecessary retention offers

In most churn scenarios, false negatives are more expensive. This typically means favouring recall — and we will see exactly how to tune this trade-off using a threshold in Section 4.4.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.5 — The precision-recall trade-off visualised
#
# This figure illustrates the concept concretely using a simulated example.
# The actual trade-off for the churn model appears in Section 4.4.
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('white')

# Left: diagram of TP/FP/FN/TN
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
ax.set_facecolor('white')

boxes = [
    {'x': 2.5, 'y': 4.2, 'w': 3.5, 'h': 1.2, 'color': '#d5f5e3', 'edge': '#27ae60',
     'label': 'True Positive (TP)', 'sub': 'Churner correctly identified'},
    {'x': 7.0, 'y': 4.2, 'w': 3.5, 'h': 1.2, 'color': '#fadbd8', 'edge': '#e74c3c',
     'label': 'False Positive (FP)', 'sub': 'Loyal customer flagged as churner'},
    {'x': 2.5, 'y': 2.5, 'w': 3.5, 'h': 1.2, 'color': '#fdebd0', 'edge': '#e67e22',
     'label': 'False Negative (FN)', 'sub': 'Churner missed by the model'},
    {'x': 7.0, 'y': 2.5, 'w': 3.5, 'h': 1.2, 'color': '#d4e6f1', 'edge': '#2980b9',
     'label': 'True Negative (TN)', 'sub': 'Loyal customer correctly ignored'},
]

for b in boxes:
    rect = mpatches.FancyBboxPatch(
        (b['x'] - b['w']/2, b['y'] - b['h']/2), b['w'], b['h'],
        boxstyle='round,pad=0.1', facecolor=b['color'], edgecolor=b['edge'], lw=2
    )
    ax.add_patch(rect)
    ax.text(b['x'], b['y'] + 0.22, b['label'], ha='center', fontsize=9, fontweight='bold')
    ax.text(b['x'], b['y'] - 0.22, b['sub'],   ha='center', fontsize=8, color='#444')

ax.text(5, 5.7, 'Predicted: Churn', ha='center', fontsize=9, color='#555', fontstyle='italic')
ax.text(5, 1.5, 'Predicted: No Churn', ha='center', fontsize=9, color='#555', fontstyle='italic')
ax.text(0.4, 4.2, 'Actual:\nChurn',    ha='center', fontsize=9, color='#555', fontstyle='italic', rotation=90)
ax.text(0.4, 2.5, 'Actual:\nNo Churn', ha='center', fontsize=9, color='#555', fontstyle='italic', rotation=90)
ax.set_title('Figure 4.5a — The Four Outcomes of a Binary Classifier',
             fontsize=10, fontweight='bold', color='#1F3864')

# Right: simulated precision-recall curve
ax2 = axes[1]
thresholds = np.linspace(0.1, 0.95, 100)
precision_sim = 0.4 + 0.55 * thresholds
recall_sim    = 0.98 - 0.85 * thresholds

ax2.plot(recall_sim, precision_sim, color='#8e44ad', lw=2.5)
# Mark three operating points
for t, label, color in [(0.2, 'High recall\n(catch more churners)', '#e74c3c'),
                         (0.5, 'Balanced', '#f39c12'),
                         (0.8, 'High precision\n(fewer false flags)', '#27ae60')]:
    idx = int(t * 99)
    ax2.scatter(recall_sim[idx], precision_sim[idx], s=120, color=color, zorder=5)
    ax2.annotate(label, xy=(recall_sim[idx], precision_sim[idx]),
                 xytext=(recall_sim[idx] - 0.05, precision_sim[idx] + 0.06),
                 fontsize=8, color=color, ha='center')

ax2.set_xlabel('Recall', fontsize=11)
ax2.set_ylabel('Precision', fontsize=11)
ax2.set_xlim(0, 1); ax2.set_ylim(0.3, 1.05)
ax2.set_title('Figure 4.5b — Precision–Recall Trade-Off\n'
              'Moving along the curve by adjusting the decision threshold',
              fontsize=10, fontweight='bold', color='#1F3864')

plt.tight_layout()
plt.show()

---
# 4.4 Business Application: Customer Churn Prediction

## The business problem

Customer churn — when a customer stops using a service — is one of the most expensive problems in subscription-based businesses. Acquiring a new customer typically costs five to ten times more than retaining an existing one. A model that identifies customers likely to leave *before they leave* gives a business the opportunity to intervene: a targeted offer, a proactive support call, a pricing adjustment.

The **Telco Customer Churn** dataset from Kaggle contains records for 7,043 telecommunications customers. Each record includes:
- Demographics (gender, age group, partner, dependents)
- Services subscribed (phone, internet, streaming, security)
- Account information (contract type, payment method, monthly charges, tenure)
- Target: whether the customer churned (`Yes`/`No`)

## Downloading the dataset

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Kaggle authentication
# Paste your Kaggle API token when prompted.
# See Chapter 0 (Section Step 4) for how to obtain your token.
# ─────────────────────────────────────────────────────────────────────────────

import os, json, getpass

kaggle_token    = getpass.getpass("Paste your Kaggle API token (KGAT_...): ")
kaggle_username = input("Enter your Kaggle username: ")

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
    json.dump({"username": kaggle_username.strip(), "key": kaggle_token.strip()}, f)
os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)

print("Kaggle credentials saved. Downloading dataset...")

!kaggle datasets download -d blastchar/telco-customer-churn --unzip -p ./data/churn

print("Download complete.")

## Exploratory Data Analysis

Before building any model, we need to understand what the data looks like. EDA answers three questions:
1. What is the class balance? (How many churners vs. non-churners?)
2. Are there missing values or data quality issues?
3. Which features are numeric and which are categorical?

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load and inspect the churn dataset
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv('./data/churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Shape: {df.shape}  ({df.shape[0]:,} customers, {df.shape[1]} columns)")
print()
print(df.dtypes)
print()
print("First 3 rows:")
df.head(3)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Class balance — the most important check before training a classifier
# ─────────────────────────────────────────────────────────────────────────────

churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

print("Class distribution:")
for cls in churn_counts.index:
    print(f"  {cls:<5}  {churn_counts[cls]:>5,}  ({churn_pct[cls]:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('white')

# Class balance bar chart
colors = ['#2980b9', '#e74c3c']
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=2)
for i, (cls, cnt) in enumerate(zip(churn_counts.index, churn_counts.values)):
    axes[0].text(i, cnt + 40, f"{cnt:,}\n({churn_pct[cls]:.1f}%)", ha='center', fontsize=10)
axes[0].set_title('Figure 4.6a — Class Imbalance', fontweight='bold', color='#1F3864')
axes[0].set_ylabel('Number of customers')
axes[0].set_ylim(0, churn_counts.max() * 1.2)

# Monthly charges by churn status
df.groupby('Churn')['MonthlyCharges'].plot(
    kind='density', ax=axes[1],
    color=['#2980b9', '#e74c3c'], lw=2.5, legend=True
)
axes[1].set_title('Figure 4.6b — Monthly Charges Distribution by Churn Status',
                  fontweight='bold', color='#1F3864')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].legend(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

## Handling Class Imbalance

The dataset has roughly 73% non-churners and 27% churners. This is a **moderately imbalanced** problem. A naive model could achieve 73% accuracy by always predicting "No Churn" — which is commercially useless.

The simplest and most effective correction for moderate imbalance is to adjust the **loss function weights**. By telling PyTorch that misclassifying a churner (the minority class) is more costly than misclassifying a loyal customer, we push the model to pay more attention to the class that matters most.

The weight for each class is set inversely proportional to its frequency: the rarer the class, the higher the weight.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Preprocessing
#
# Steps:
#   1. Fix TotalCharges — it is stored as string; convert to numeric
#   2. Drop customerID — a unique identifier with no predictive value
#   3. Encode all categorical columns with LabelEncoder
#   4. Scale all numeric columns with StandardScaler
#   5. Split into train/validation/test sets
#   6. Compute class weights for the loss function
# ─────────────────────────────────────────────────────────────────────────────

df_clean = df.copy()

# TotalCharges has a few blank strings — replace with 0, then convert to float
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0.0)

# Drop the ID column
df_clean.drop(columns=['customerID'], inplace=True)

# Encode target
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)

# Encode all other categorical columns
categorical_cols = df_clean.select_dtypes(include='object').columns.tolist()
label_encoders   = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le

# Separate features and target
X = df_clean.drop(columns=['Churn']).values.astype(np.float32)
y = df_clean['Churn'].values.astype(np.int64)

# Train / validation / test split (70 / 15 / 15)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Scale numeric features using training statistics only
# (fitting on the full dataset would leak information from val/test)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Training:   {X_train.shape[0]:,} samples")
print(f"Validation: {X_val.shape[0]:,} samples")
print(f"Test:       {X_test.shape[0]:,} samples")
print(f"Features:   {X_train.shape[1]}")

# ── Class weights ────────────────────────────────────────────────────────────
# Weight for each class = total_samples / (n_classes * samples_in_class)
# The minority class (churn=1) gets a higher weight
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
weight_pos = len(y_train) / (2 * n_pos)   # weight for churn=1
weight_neg = len(y_train) / (2 * n_neg)   # weight for churn=0

class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float32).to(device)

print(f"\nClass weights:  No Churn = {weight_neg:.3f}  |  Churn = {weight_pos:.3f}")
print("(Higher weight for the minority churn class — the model penalises missing churners more.)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build PyTorch DataLoaders for the churn dataset
# ─────────────────────────────────────────────────────────────────────────────

def make_loader(X, y, batch_size=64, shuffle=False):
    """Wrap numpy arrays into a PyTorch DataLoader."""
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

churn_train_loader = make_loader(X_train, y_train, shuffle=True)
churn_val_loader   = make_loader(X_val,   y_val)
churn_test_loader  = make_loader(X_test,  y_test)

print("DataLoaders created.")
print(f"Training batches: {len(churn_train_loader)}")

## Training the Churn Classifier

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Define the churn model
#
# Input:  19 features (after encoding and dropping customerID)
# Hidden: three layers — progressively narrowing
# Output: 2 logits (for CrossEntropyLoss: No Churn / Churn)
# ─────────────────────────────────────────────────────────────────────────────

NUM_FEATURES = X_train.shape[1]

churn_model = FFN(
    input_size   = NUM_FEATURES,
    hidden_sizes = [128, 64, 32],
    output_size  = 2,
    dropout_p    = 0.3
).to(device)

# Weighted loss function — penalises missing churners more heavily
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(churn_model.parameters(), lr=1e-3)

# Learning rate scheduler: reduce LR if validation loss plateaus
# patience=5 means: if val_loss has not improved for 5 epochs, reduce LR by 50%
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(churn_model)
print(f"\nTrainable parameters: {count_parameters(churn_model):,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Training loop with early stopping
#
# Early stopping halts training when the validation loss has not improved
# for a set number of epochs. This prevents overfitting and saves compute.
#
# We keep a copy of the best model weights seen during training.
# ─────────────────────────────────────────────────────────────────────────────

import copy

MAX_EPOCHS     = 60
EARLY_STOP_PAT = 10   # stop if val_loss has not improved for 10 epochs

churn_history  = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_loss  = float('inf')
best_weights   = None
patience_count = 0

for epoch in range(1, MAX_EPOCHS + 1):

    # ── Training ────────────────────────────────────────────────────────────
    churn_model.train()
    running_loss = 0.0

    for X_batch, y_batch in churn_train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = churn_model(X_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(churn_train_loader)

    # ── Validation ──────────────────────────────────────────────────────────
    churn_model.eval()
    val_loss = 0.0; correct = 0; total = 0

    with torch.no_grad():
        for X_batch, y_batch in churn_val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs  = churn_model(X_batch)
            val_loss += criterion(outputs, y_batch).item()
            predicted = outputs.argmax(dim=1)
            correct  += (predicted == y_batch).sum().item()
            total    += y_batch.size(0)

    val_loss /= len(churn_val_loader)
    val_acc   = 100.0 * correct / total

    scheduler.step(val_loss)

    churn_history['train_loss'].append(train_loss)
    churn_history['val_loss'].append(val_loss)
    churn_history['val_acc'].append(val_acc)

    # ── Early stopping ──────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights  = copy.deepcopy(churn_model.state_dict())
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= EARLY_STOP_PAT:
            print(f"Early stopping at epoch {epoch}.")
            break

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}  Train: {train_loss:.4f}  Val: {val_loss:.4f}  Acc: {val_acc:.1f}%")

# Restore best weights before evaluation
churn_model.load_state_dict(best_weights)
print(f"\nBest validation loss: {best_val_loss:.4f}")
print("Best model weights restored.")

## Full Evaluation

### Confusion matrix, precision, recall, F1

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Collect predictions on the held-out test set
# ─────────────────────────────────────────────────────────────────────────────

churn_model.eval()
test_preds  = []
test_probs  = []   # probability of churn (class=1), used for ROC
test_labels = []

with torch.no_grad():
    for X_batch, y_batch in churn_test_loader:
        X_batch = X_batch.to(device)
        logits  = churn_model(X_batch)
        probs   = torch.softmax(logits, dim=1)[:, 1]   # P(churn)
        preds   = logits.argmax(dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_probs.extend(probs.cpu().numpy())
        test_labels.extend(y_batch.numpy())

test_preds  = np.array(test_preds)
test_probs  = np.array(test_probs)
test_labels = np.array(test_labels)

print("Classification Report:")
print(classification_report(test_labels, test_preds, target_names=['No Churn', 'Churn']))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4.7 — Full evaluation dashboard
#   (a) Training curves   (b) Confusion matrix
#   (c) ROC curve         (d) Cost analysis
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.patch.set_facecolor('white')

# ── (a) Training curves ─────────────────────────────────────────────────────
ax = axes[0, 0]
ep = range(1, len(churn_history['train_loss']) + 1)
ax.plot(ep, churn_history['train_loss'], color='#2980b9', lw=2.5, label='Train loss')
ax.plot(ep, churn_history['val_loss'],   color='#e74c3c', lw=2.5, linestyle='--', label='Val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('(a) Loss Curves', fontweight='bold', color='#1F3864')
ax.legend()

# ── (b) Confusion matrix ─────────────────────────────────────────────────────
ax = axes[0, 1]
cm_churn = confusion_matrix(test_labels, test_preds)
sns.heatmap(
    cm_churn, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No Churn', 'Churn'],
    yticklabels=['No Churn', 'Churn'],
    linewidths=1, ax=ax
)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('(b) Confusion Matrix — Test Set', fontweight='bold', color='#1F3864')

# ── (c) ROC curve ────────────────────────────────────────────────────────────
ax = axes[1, 0]
fpr, tpr, thresholds_roc = roc_curve(test_labels, test_probs)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color='#8e44ad', lw=2.5, label=f'ROC AUC = {roc_auc:.3f}')
ax.plot([0, 1], [0, 1], color='#aaaaaa', lw=1.5, linestyle='--', label='Random classifier')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('(c) ROC Curve', fontweight='bold', color='#1F3864')
ax.legend(loc='lower right')

# ── (d) Cost analysis — threshold sweep ─────────────────────────────────────
# Business assumptions (illustrative):
#   False negative cost: $200 (losing a churned customer — lost lifetime value)
#   False positive cost: $20  (unnecessary retention offer sent to a loyal customer)
ax = axes[1, 1]
FN_COST = 200
FP_COST = 20

sweep_thresholds = np.linspace(0.1, 0.9, 80)
total_costs = []

for t in sweep_thresholds:
    preds_t = (test_probs >= t).astype(int)
    cm_t    = confusion_matrix(test_labels, preds_t, labels=[0, 1])
    tn, fp, fn, tp = cm_t.ravel()
    cost = fn * FN_COST + fp * FP_COST
    total_costs.append(cost)

best_t_idx  = np.argmin(total_costs)
best_t      = sweep_thresholds[best_t_idx]
best_cost   = total_costs[best_t_idx]

ax.plot(sweep_thresholds, total_costs, color='#e67e22', lw=2.5)
ax.axvline(best_t, color='#c0392b', lw=2, linestyle='--')
ax.scatter([best_t], [best_cost], color='#c0392b', s=120, zorder=5)
ax.annotate(
    f"Optimal threshold: {best_t:.2f}\nTotal cost: ${best_cost:,}",
    xy=(best_t, best_cost), xytext=(best_t + 0.08, best_cost + 500),
    fontsize=9, color='#c0392b',
    arrowprops=dict(arrowstyle='->', color='#c0392b')
)
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Total business cost ($)')
ax.set_title(f'(d) Cost Analysis (FN cost=${FN_COST}, FP cost=${FP_COST})',
             fontweight='bold', color='#1F3864')

fig.suptitle('Figure 4.7 — Churn Model: Full Evaluation Dashboard',
             fontsize=14, fontweight='bold', color='#1F3864', y=1.01)
plt.tight_layout()
plt.show()

print(f"\nROC AUC: {roc_auc:.4f}")
print(f"Business-optimal threshold: {best_t:.2f}  (minimises FN×$200 + FP×$20)")
print(f"Estimated total cost at optimal threshold: ${best_cost:,}")

### 📝 Exercise 4.4 — Adjust the Cost Assumptions

The cost analysis above uses illustrative values ($200 per missed churner, $20 per false flag). Your company's actual costs may differ significantly.

Change the `FN_COST` and `FP_COST` values below and observe how the optimal threshold shifts. Think about what this means operationally.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 4.4 — Rerun the cost analysis with your own cost assumptions
#
# Scenario: your company has a very high-touch retention team.
#   - Each retention call costs $80 (higher FP cost)
#   - Each lost customer represents $500 in lost value (higher FN cost)
#
# Adjust the values below. Does the optimal threshold go up or down?
# What does that tell you about when to flag a customer for intervention?
# ─────────────────────────────────────────────────────────────────────────────

FN_COST_EX = ?   # replace with your value
FP_COST_EX = ?   # replace with your value

total_costs_ex = []
for t in sweep_thresholds:
    preds_t = (test_probs >= t).astype(int)
    cm_t    = confusion_matrix(test_labels, preds_t, labels=[0, 1])
    tn, fp, fn, tp = cm_t.ravel()
    total_costs_ex.append(fn * FN_COST_EX + fp * FP_COST_EX)

best_t_ex   = sweep_thresholds[np.argmin(total_costs_ex)]
best_cost_ex = min(total_costs_ex)

print(f"Your cost assumptions: FN=${FN_COST_EX}, FP=${FP_COST_EX}")
print(f"Optimal threshold:     {best_t_ex:.2f}")
print(f"Estimated total cost:  ${best_cost_ex:,}")
print()
print("Reflection: compared to the original threshold, did your assumptions")
print("push the model to be more aggressive (flag more) or more conservative?")

---
# 4.5 Pipeline: Saving and Deploying the Churn Model

## Revisiting the ModelPipeline

In Chapter 3 we built the `ModelPipeline` class — a production-ready wrapper that bundles everything a trained model needs to be useful outside the training notebook. We use exactly the same class here, configured for the churn problem.

As a reminder, what `ModelPipeline` saves into a single `.pth` file:

| Component | What it stores |
|-----------|----------------|
| **Architecture config** | `input_size`, `hidden_sizes`, `output_size`, `dropout_p` — enough to reconstruct the `FFN` |
| **Model weights** | The trained `state_dict` — no class dependency |
| **Model class** | Serialised with `dill` so it can be reconstructed on any machine without training code |
| **Preprocessor** | The `StandardScaler` fitted on training data |
| **Feature names** | Column names in expected order — used for input validation |
| **Feature ranges** | Min/max per column — used to catch out-of-range values at inference time |
| **Training history** | Loss curves — preserved across retraining sessions |

The only addition specific to classification is `class_labels` — the mapping from output index to business label (0 → No Churn, 1 → Churn) — which we pass in via `model_config`.

In any subsequent session — on any machine — one call to `ModelPipeline.load()` reconstructs the fully working pipeline. No training code required.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ModelPipeline — same class as Chapter 3, configured for classification.
#
# What is the same as ch03:
#   - dill serialises the FFN class so load() needs no training code
#   - validate_input() checks columns, nulls, AND value ranges
#   - training_history is saved and extended on every retrain()
#   - save / load / validate_input / predict / retrain API is identical
#
# What is added for classification:
#   - model_config includes class_labels ({0: 'No Churn', 1: 'Churn'})
#   - predict() accepts a threshold param and returns a labelled DataFrame
#   - predict() applies softmax and returns P(churn) alongside the label
# ─────────────────────────────────────────────────────────────────────────────

import dill, datetime

class ModelPipeline:
    """Bundles a trained PyTorch model with its preprocessor and metadata.
    Provides: save | load | validate_input | predict | retrain.
    Identical contract to the ModelPipeline introduced in Chapter 3.
    """

    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model             = model
        self.model_config      = model_config      # dict of constructor kwargs + class_labels
        self.preprocessor      = preprocessor      # fitted StandardScaler
        self.feature_names     = feature_names or []
        self.feature_ranges    = feature_ranges or {}   # {col: (min, max)} — for range checks
        self._model_class      = model_class
        self.model_class       = model_class.__name__ if model_class else type(model).__name__
        self.training_history  = {'train_loss': [], 'val_loss': []}

    # ── Save ──────────────────────────────────────────────────────────────────
    def save(self, path):
        """Save the complete pipeline to a single .pth file.
        dill serialises the FFN class itself so load() works on any machine
        without importing or redefining the class.
        """
        cls_obj     = self._model_class if self._model_class is not None else type(self.model)
        class_bytes = dill.dumps(cls_obj)
        checkpoint  = {
            'model_class'       : self.model_class,
            'model_class_bytes' : class_bytes,
            'model_config'      : self.model_config,
            'state_dict'        : self.model.state_dict(),
            'preprocessor'      : self.preprocessor,
            'feature_names'     : self.feature_names,
            'feature_ranges'    : self.feature_ranges,
            'training_history'  : self.training_history,
            'pytorch_version'   : torch.__version__,
            'saved_at'          : datetime.datetime.now().isoformat(),
        }
        torch.save(checkpoint, path)
        print(f'Pipeline saved → {path}')
        print(f'  model_class : {self.model_class}')
        print(f'  features    : {len(self.feature_names)}')
        print(f'  history     : {len(self.training_history["val_loss"])} epochs recorded')
        print(f'  saved_at    : {checkpoint["saved_at"]}')

    # ── Load ──────────────────────────────────────────────────────────────────
    @classmethod
    def load(cls, path, device=None):
        """Load a saved pipeline. No training code or class definition needed.
        The FFN class is reconstructed from the dill bytes inside the .pth file.
        """
        device     = device or torch.device('cpu')
        checkpoint = torch.load(path, map_location=device, weights_only=False)

        # Reconstruct the model class from dill bytes — works in notebooks and .py files
        model_class = dill.loads(checkpoint['model_class_bytes'])

        # Build architecture from saved config, then load weights
        cfg   = checkpoint['model_config']
        model = model_class(
            input_size   = cfg['input_size'],
            hidden_sizes = cfg['hidden_sizes'],
            output_size  = cfg['output_size'],
            dropout_p    = cfg['dropout_p'],
        )
        model.load_state_dict(checkpoint['state_dict'])
        model.to(device)
        model.eval()

        pipeline = cls(
            model          = model,
            model_config   = cfg,
            preprocessor   = checkpoint.get('preprocessor'),
            feature_names  = checkpoint.get('feature_names', []),
            feature_ranges = checkpoint.get('feature_ranges', {}),
            model_class    = model_class,
        )
        pipeline.training_history = checkpoint.get(
            'training_history', {'train_loss': [], 'val_loss': []})

        print(f'Pipeline loaded ← {path}')
        print(f'  model_class     : {checkpoint.get("model_class")}  (reconstructed from saved class)')
        print(f'  saved_at        : {checkpoint.get("saved_at")}')
        print(f'  pytorch_version : {checkpoint.get("pytorch_version")}')
        return pipeline

    # ── Validate input ─────────────────────────────────────────────────────────
    def validate_input(self, df):
        """Check incoming DataFrame against the schema recorded at training time.
        Checks: required columns present · no nulls · values within training ranges.
        Raises ValueError with an actionable message on failure.
        """
        # 1. Column presence
        missing = set(self.feature_names) - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")
        extra = set(df.columns) - set(self.feature_names)
        if extra:
            print(f"Warning: extra columns ignored: {sorted(extra)}")

        df_check = df[self.feature_names]

        # 2. Null check
        null_cols = df_check.columns[df_check.isnull().any()].tolist()
        if null_cols:
            raise ValueError(f"Null values found in: {null_cols}. Fill or drop before predicting.")

        # 3. Range check — warn on out-of-range values, raise if severe (>10× training range)
        for col, (lo, hi) in self.feature_ranges.items():
            if col not in df_check.columns:
                continue
            col_min = df_check[col].min()
            col_max = df_check[col].max()
            span    = hi - lo if hi != lo else 1.0
            if col_min < lo - 10 * span or col_max > hi + 10 * span:
                raise ValueError(
                    f"Column '{col}' has values [{col_min:.2f}, {col_max:.2f}] "
                    f"far outside training range [{lo:.2f}, {hi:.2f}]. "
                    f"Check for data pipeline errors."
                )

        return True

    # ── Predict ────────────────────────────────────────────────────────────────
    def predict(self, df, threshold=0.5):
        """Predict churn for new customer records.

        Parameters
        ----------
        df        : DataFrame with the same features used during training (raw, unscaled)
        threshold : float — probability above which a customer is flagged as churner

        Returns
        -------
        DataFrame with columns: churn_probability, predicted_label
        """
        self.validate_input(df)
        X      = df[self.feature_names].values.astype(np.float32)
        X      = self.preprocessor.transform(X)   # apply the saved scaler
        tensor = torch.tensor(X, dtype=torch.float32)

        self.model.eval()
        with torch.no_grad():
            probs = torch.softmax(self.model(tensor), dim=1)[:, 1].numpy()

        class_labels = self.model_config.get('class_labels', {0: '0', 1: '1'})
        labels = [class_labels[int(p >= threshold)] for p in probs]
        return pd.DataFrame({
            'churn_probability': np.round(probs, 4),
            'predicted_label':   labels,
        })

    # ── Retrain ────────────────────────────────────────────────────────────────
    def retrain(self, X_new, y_new, epochs=10, lr=1e-4):
        """Continue training on a new batch of labelled data.

        X_new : raw (unscaled) numpy array — the scaler is applied internally
        y_new : integer class labels (0 = No Churn, 1 = Churn)
        lr    : low learning rate (default 1e-4) to preserve existing knowledge

        Training loss is appended to self.training_history on every call,
        so the full history is preserved when the pipeline is saved.
        """
        X_scaled = self.preprocessor.transform(X_new.astype(np.float32))
        loader   = make_loader(X_scaled, y_new, shuffle=True)
        opt      = optim.Adam(self.model.parameters(), lr=lr)
        crit     = nn.CrossEntropyLoss()

        self.model.train()
        for epoch in range(1, epochs + 1):
            epoch_loss = 0.0
            for X_batch, y_batch in loader:
                opt.zero_grad()
                loss = crit(self.model(X_batch), y_batch)
                loss.backward()
                opt.step()
                epoch_loss += loss.item()
            avg_loss = epoch_loss / len(loader)
            self.training_history['train_loss'].append(avg_loss)  # accumulate history
            print(f"  Retrain epoch {epoch}/{epochs}  loss: {avg_loss:.4f}")

        self.model.eval()
        print(f"Retraining complete. Total epochs in history: {len(self.training_history['train_loss'])}")

print("ModelPipeline class defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Wrap the trained model in a ModelPipeline and save it
# ─────────────────────────────────────────────────────────────────────────────

feature_names = df_clean.drop(columns=['Churn']).columns.tolist()

# Compute per-feature min/max from the raw (unscaled) training data
# These are stored in the pipeline and used by validate_input() at inference time
X_train_raw     = scaler.inverse_transform(X_train)
feature_ranges  = {
    col: (float(X_train_raw[:, i].min()), float(X_train_raw[:, i].max()))
    for i, col in enumerate(feature_names)
}

# class_labels lives inside model_config so it is saved and restored automatically
model_config = {
    'input_size':   NUM_FEATURES,
    'hidden_sizes': [128, 64, 32],
    'output_size':  2,
    'dropout_p':    0.3,
    'class_labels': {0: 'No Churn', 1: 'Churn'},   # restored by load() → used in predict()
}

pipeline = ModelPipeline(
    model          = churn_model,
    model_config   = model_config,
    preprocessor   = scaler,
    feature_names  = feature_names,
    feature_ranges = feature_ranges,
    model_class    = FFN,
)

# Copy the training history recorded during training into the pipeline
pipeline.training_history['train_loss'] = churn_history['train_loss']
pipeline.training_history['val_loss']   = churn_history['val_loss']

pipeline.save('churn_model_v1.pth')
print("\nThe .pth file contains everything needed to score new customers.")
print("Share it with any team member — no training code required.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load the pipeline in a clean context and score new customers
#
# In practice this would be in a separate notebook or script.
# Here we simulate it by deleting the pipeline object and reloading from disk.
# The FFN class is reconstructed from the dill bytes inside the .pth file —
# no FFN definition or training code is needed.
# ─────────────────────────────────────────────────────────────────────────────

del pipeline   # simulate a fresh session

loaded_pipeline = ModelPipeline.load('churn_model_v1.pth')
print()
print(f"Features expected : {loaded_pipeline.feature_names}")
print(f"Training history  : {len(loaded_pipeline.training_history['val_loss'])} epochs")

# ── Score 5 customers from the test set ─────────────────────────────────────
# predict() expects raw (unscaled) data — it applies the scaler internally,
# exactly as ch03 ModelPipeline.predict() does.
new_customers_raw = pd.DataFrame(
    scaler.inverse_transform(X_test[:5]), columns=feature_names
)

results = loaded_pipeline.predict(new_customers_raw, threshold=best_t)
results['actual'] = ['Churn' if l == 1 else 'No Churn' for l in y_test[:5]]

print(f"\nPredictions (threshold = {best_t:.2f}):")
print(results.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Demonstrate input validation — three failure cases, matching ch03 §3.7f
# ─────────────────────────────────────────────────────────────────────────────

print("Case 1 — missing required column:")
bad_df = new_customers_raw.drop(columns=[feature_names[0]])
try:
    loaded_pipeline.predict(bad_df)
except ValueError as e:
    print(f"  ✓ Caught: {e}")

print()
print("Case 2 — null value in a column:")
null_df = new_customers_raw.copy()
null_df.iloc[0, 1] = np.nan
try:
    loaded_pipeline.predict(null_df)
except ValueError as e:
    print(f"  ✓ Caught: {e}")

print()
print("Case 3 — values wildly outside training range:")
oor_df = new_customers_raw.copy()
oor_df[feature_names[0]] = 999999.0   # inject an absurd value
try:
    loaded_pipeline.predict(oor_df)
except ValueError as e:
    print(f"  ✓ Caught: {e}")

print()
print("Case 4 — valid input (all checks pass):")
result = loaded_pipeline.validate_input(new_customers_raw)
print(f"  validate_input returned: {result}  ✓")
print("The pipeline protects downstream systems from silent failures.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Retraining on a new month of data
#
# Scenario: one month has passed. The CRM has labelled an additional 500
# customers with ground-truth churn outcomes. We update the model.
#
# We simulate this using the validation set as 'new' data.
# A low learning rate (1e-4) prevents the model from forgetting what it learned.
# ─────────────────────────────────────────────────────────────────────────────

print("Simulating one month of new labelled data (using val set as proxy)...")
print()

loaded_pipeline.retrain(X_val, y_val, epochs=5, lr=1e-4)

# Save the updated model as v2
loaded_pipeline.save('churn_model_v2.pth')
print("\nVersion 2 saved. The original v1 is preserved for rollback if needed.")

---
## Chapter 4 Summary

| Concept | Key takeaway |
|---------|-------------|
| **FFN architecture** | `nn.Module` with stacked `Linear → ReLU → Dropout` blocks; `output_size` matches the task |
| **`nn.Sequential`** | A clean way to chain layers without writing a custom `forward` for each block |
| **MNIST** | A 10-class classifier trained with `CrossEntropyLoss`; accuracy alone is a reliable metric when classes are balanced |
| **Confusion matrix** | Reveals *where* a model fails — essential for understanding misclassification patterns |
| **Imbalanced classes** | `CrossEntropyLoss(weight=...)` re-weights the loss so the minority class receives proportionally more attention |
| **Precision / Recall / F1** | Three complementary metrics that expose trade-offs hidden by accuracy |
| **ROC-AUC** | A threshold-independent measure of separability between classes |
| **Cost analysis** | Translating model errors into business costs reveals the optimal decision threshold |
| **Early stopping** | Halts training when validation loss plateaus, preventing overfitting and wasted compute |
| **ModelPipeline** | Same class from Chapter 3 — saves model, preprocessor, feature ranges, and training history into one `.pth` file |

---

## What Comes Next

Chapter 5 introduces **Convolutional Neural Networks (CNNs)** — the architecture designed for image data. Where an FFN treats every pixel as an independent input, a CNN learns to recognise local patterns (edges, textures, shapes) regardless of where they appear in the image. The business application is manufacturing defect detection: training a model to classify product images as conforming or defective from a Kaggle dataset of casting process images.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*